# Python counterpart to MATLAB BRCM validation

This notebook is the Python counterpart of `matlab_validation/run_brcm_end_to_end.m`. It uses the same IDF, empty-EHF BuildingModel, identifier order, sampling time, initial state, ambient-boundary feedback, and simulation horizon.

It exports logically matching CSV, JSON, and MAT files for direct MATLAB-vs-Python comparison. Rows and identifiers are never reordered.

> `BoundaryCondition.value` is treated as conductance `G` because MATLAB `generateThermalModel.m` assigns it as inverse total resistance. The simulation input is `q = G(Tamb - T_boundary_state)`.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from scipy.io import loadmat, savemat

def find_repository(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'brcm').is_dir():
            return candidate
    raise RuntimeError('Repository root containing src/brcm was not found')

REPO = find_repository(Path.cwd().resolve())
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

import brcm
from brcm.energyplus import audit_conversion, conversion_to_thermal_model_data

IDF_PATH = REPO / 'pre_test/tests/fixtures/energyplus/representative_multizone.idf'
PYTHON_OUTPUT = REPO / 'python_validation/outputs'
MATLAB_OUTPUT = REPO / 'matlab_validation/outputs'

INITIAL_TEMPERATURE_C = 20.0
AMBIENT_TEMPERATURE_C = 30.0
ZONE_HEATING_W = 0.0
SAMPLE_TIME_HOURS = 1 / 60
N_STEPS = 60

for folder in ('model_config/tables', 'matrices', 'simulation'):
    (PYTHON_OUTPUT / folder).mkdir(parents=True, exist_ok=True)
print('IDF:', IDF_PATH)
print('Python output:', PYTHON_OUTPUT)
print('MATLAB output available:', (MATLAB_OUTPUT / 'manifest.json').exists())

## 1. Generate identical logical model inputs

The seven DataFrames below retain conversion row order and exact schema order. They are the canonical table representation used for export and later comparison.

In [ ]:
conversion = brcm.convert_idf_to_brcm_data(IDF_PATH)
data = conversion_to_thermal_model_data(conversion)
thermal = brcm.generate_thermal_model(data)
audit = audit_conversion(conversion, thermal)

table_frames = {
    name: pd.DataFrame(rows[1:], columns=rows[0])
    for name, rows in conversion.tables.items()
}
for name, frame in table_frames.items():
    frame.to_csv(PYTHON_OUTPUT / 'model_config/tables' / f'{name}.csv', sep=';', index=False, lineterminator='\n')
pd.DataFrame({'rows': {name: len(frame) for name, frame in table_frames.items()}})

## 2. Thermal and empty-EHF BuildingModel

An EnergyPlus IDF does not declare BRCM EHF models, so both validation workflows compose a BuildingModel with no EHF declarations. This produces zero-width `u`, `v`, `y`, and constraint axes while retaining the thermal dynamics.

In [ ]:
building = brcm.BuildingModel(thermal, [])
building.discretize(SAMPLE_TIME_HOURS)
Fx, Fu, Fv, g, constraint_ids = building.get_constraints_matrices({})
cu = building.get_cost_vector({})

ids = {
    'x': list(building.identifiers.x), 'q': list(building.identifiers.q),
    'u': list(building.identifiers.u), 'v': list(building.identifiers.v),
    'y': list(building.identifiers.y),
    'constraints': list(building.identifiers.constraints),
}
with (PYTHON_OUTPUT / 'model_config/identifiers.json').open('w') as handle:
    json.dump(ids, handle, indent=2)

boundary_records = [
    {'type': kind, 'identifier_1': item.identifier_1,
     'identifier_2': item.identifier_2, 'value': item.value}
    for kind, items in thermal.boundary_conditions.items() for item in items
]
with (PYTHON_OUTPUT / 'model_config/boundary_conditions.json').open('w') as handle:
    json.dump(boundary_records, handle, indent=2)

thermal_Ad, thermal_Bqd = brcm.BuildingModel._zoh(
    thermal.A, thermal.Bq, SAMPLE_TIME_HOURS * 3600
)
savemat(PYTHON_OUTPUT / 'matrices/thermal_model.mat', {
    'thermal_A': thermal.A, 'thermal_Bq': thermal.Bq,
    'thermal_Xcap': thermal.Xcap, 'thermal_Ad': thermal_Ad,
    'thermal_Bqd': thermal_Bqd, 'thermal_A_d': thermal_Ad,
    'thermal_Bq_d': thermal_Bqd,
})
c, d = building.continuous_time_model, building.discrete_time_model
savemat(PYTHON_OUTPUT / 'matrices/building_continuous.mat', {
    name: getattr(c, name) for name in ('A','Bu','Bv','Bxu','Bvu','C','Du','Dv','Dxu','Dvu')
})
savemat(PYTHON_OUTPUT / 'matrices/building_discrete.mat', {
    'Ad': d.A, 'Bdu': d.Bu, 'Bdv': d.Bv, 'Bdxu': d.Bxu, 'Bdvu': d.Bvu,
    'Cd': d.C, 'Ddu': d.Du, 'Ddv': d.Dv, 'Ddxu': d.Dxu, 'Ddvu': d.Dvu,
})
savemat(PYTHON_OUTPUT / 'matrices/constraints_cost.mat',
        {'Fx': Fx, 'Fu': Fu, 'Fv': Fv, 'g': g, 'cu': cu})

## 3. Identical deterministic simulation

Python `X` already follows the MATLAB-compatible `N`-column convention. `X_full` retains the post-input state at column `N+1`. The actual generated `Q` is exported.

In [ ]:
state_index = {name: i for i, name in enumerate(ids['x'])}
q_index = {name: i for i, name in enumerate(ids['q'])}
ambient_boundaries = [item for item in boundary_records if item['type'] == 'ambient']
zone_state = next(name for name in ids['x'] if name.startswith('x_Z'))

def inputs(current_x, _time_hours, identifiers):
    q = np.zeros((len(identifiers.q), 1))
    for boundary in ambient_boundaries:
        state = boundary['identifier_1'] if boundary['identifier_1'] in state_index else boundary['identifier_2']
        q_name = 'q' + state[1:]
        q[q_index[q_name], 0] += boundary['value'] * (
            AMBIENT_TEMPERATURE_C - current_x[state_index[state], 0]
        )
    q[q_index['q' + zone_state[1:]], 0] += ZONE_HEATING_W
    return q

x0 = np.full((len(ids['x']), 1), INITIAL_TEMPERATURE_C)
simulation = brcm.simulate_tm(thermal, SAMPLE_TIME_HOURS, x0, N_STEPS, inputs)
repeat = brcm.simulate_tm(thermal, SAMPLE_TIME_HOURS, x0, N_STEPS, inputs)
deterministic_repeat = np.array_equal(simulation.X_full, repeat.X_full)
U = np.zeros((len(ids['u']), N_STEPS))
V = np.zeros((len(ids['v']), N_STEPS))
Y = np.zeros((len(ids['y']), N_STEPS))
zone_mask = np.array([name.startswith('x_Z') for name in ids['x']])
savemat(PYTHON_OUTPUT / 'simulation/simulation_inputs.mat', {
    'x0': x0, 'Ts_hrs': SAMPLE_TIME_HOURS, 'n_steps': N_STEPS,
    'Q': simulation.Q, 'U': U, 'V': V,
})
savemat(PYTHON_OUTPUT / 'simulation/simulation_results.mat', {
    'X': simulation.X, 'X_full': simulation.X_full, 'Y': Y,
    'Q': simulation.Q, 'U': U, 'V': V, 't_hrs': simulation.t_hrs,
    'representative_state_trajectories': simulation.X_full[zone_mask],
    'state_min': simulation.X_full.min(axis=1, keepdims=True),
    'state_max': simulation.X_full.max(axis=1, keepdims=True),
    'state_final': simulation.X_full[:, -1:],
})
simulation_summary = {
    'matlab_compatible_X_shape': list(simulation.X.shape),
    'full_state_shape': list(simulation.X_full.shape),
    'Q_shape': list(simulation.Q.shape), 'U_shape': list(U.shape),
    'V_shape': list(V.shape), 'Y_shape': list(Y.shape),
    'minimum_state': float(simulation.X_full.min()),
    'maximum_state': float(simulation.X_full.max()),
    'final_state': simulation.X_full[:, -1].tolist(),
    'deterministic_repeat': deterministic_repeat,
}
simulation_summary

## 4. Write compatible metadata and manifest

In [ ]:
simulation_config = {
    'INITIAL_TEMPERATURE_C': INITIAL_TEMPERATURE_C,
    'AMBIENT_TEMPERATURE_C': AMBIENT_TEMPERATURE_C,
    'ZONE_HEATING_W': ZONE_HEATING_W,
    'SAMPLE_TIME_HOURS': SAMPLE_TIME_HOURS, 'N_STEPS': N_STEPS,
    'input_rule': 'For each ambient boundary: q = G * (Tamb - T_boundary_state)',
    'boundary_value_semantics': 'Conductance G [W/K]',
}
matrix_axes = {
    'thermal_A':['x','x'], 'thermal_Bq':['x','q'], 'thermal_Xcap':['x','x'],
    'thermal_Ad':['x','x'], 'thermal_Bqd':['x','q'],
    'A':['x','x'], 'Bu':['x','u'], 'Bv':['x','v'],
    'Bxu':['x','x','u'], 'Bvu':['x','v','u'],
    'C':['y','x'], 'Du':['y','u'], 'Dv':['y','v'],
    'Dxu':['y','x','u'], 'Dvu':['y','v','u'],
}
metadata = {
    'source_idf': str(IDF_PATH.relative_to(REPO)), 'energyplus_version': audit.energyplus_version,
    'idd_file': 'V8-1-0-Energy+.idd', 'sampling_time_hours': SAMPLE_TIME_HOURS,
    'initial_state_definition': 'Uniform 20 degC for every thermal state',
    'ehf_declarations': [],
    'building_model_note': 'BuildingModel generated with no EHF declarations.',
}
for filename, value in {
    'model_config/metadata.json': metadata, 'model_config/matrix_axes.json': matrix_axes,
    'model_config/constraint_identifiers.json': constraint_ids,
    'simulation/simulation_config.json': simulation_config,
    'simulation/simulation_summary.json': simulation_summary,
}.items():
    with (PYTHON_OUTPUT / filename).open('w') as handle:
        json.dump(value, handle, indent=2)

manifest = {
    'schema_version': 1, 'source_idf': str(IDF_PATH.relative_to(REPO)),
    'runtime': f'Python {sys.version.split()[0]}',
    'stages': {name: 'PASS' for name in (
        'conversion','tables','thermal_model','building_model',
        'discretization','simulation','repeatability')},
    'model_dimensions': {
        'n_x':len(ids['x']), 'n_q':len(ids['q']), 'n_u':len(ids['u']),
        'n_v':len(ids['v']), 'n_y':len(ids['y']),
        'n_constraints':len(ids['constraints'])},
    'simulation_settings': simulation_config, 'simulation_summary': simulation_summary,
    'warnings': list(conversion.warnings), 'errors': [], 'overall_status': 'PASS',
}
with (PYTHON_OUTPUT / 'manifest.json').open('w') as handle:
    json.dump(manifest, handle, indent=2)
print('Python comparable export: PASS')
print(PYTHON_OUTPUT / 'manifest.json')

## 5. DataFrame and numerical comparison when MATLAB output exists

The comparison keeps table rows in source order. Only empty/NaN representation and numerical string formatting are normalized. A missing MATLAB manifest leaves this section in `WAITING` state.

In [ ]:
def normalized_frame(path):
    frame = pd.read_csv(path, sep=';', dtype=str, keep_default_na=False)
    frame = frame.loc[:, [name for name in frame.columns if not name.startswith('Unnamed:')]]
    return frame.replace({'NaN':'', 'nan':''})

comparison_rows = []
if (MATLAB_OUTPUT / 'manifest.json').exists():
    for name, python_frame in table_frames.items():
        matlab_frame = normalized_frame(MATLAB_OUTPUT / 'model_config/tables' / f'{name}.csv')
        python_normalized = normalized_frame(PYTHON_OUTPUT / 'model_config/tables' / f'{name}.csv')
        comparison_rows.append({
            'quantity': f'table:{name}', 'shape_match': matlab_frame.shape == python_normalized.shape,
            'ordered_values_match': matlab_frame.equals(python_normalized),
        })
    matlab_tm = loadmat(MATLAB_OUTPUT / 'matrices/thermal_model.mat')
    python_tm = loadmat(PYTHON_OUTPUT / 'matrices/thermal_model.mat')
    for name in ('thermal_A','thermal_Bq','thermal_Xcap','thermal_Ad','thermal_Bqd'):
        left, right = matlab_tm[name], python_tm[name]
        comparison_rows.append({
            'quantity': name, 'shape_match': left.shape == right.shape,
            'ordered_values_match': bool(np.allclose(left, right, rtol=1e-10, atol=1e-12)),
        })
    comparison_status = pd.DataFrame(comparison_rows).set_index('quantity')
else:
    comparison_status = pd.DataFrame([{'quantity':'MATLAB outputs', 'shape_match':False,
                                       'ordered_values_match':'WAITING'}]).set_index('quantity')
comparison_status